In [14]:
import sys
sys.path.insert(0, "/home/ibm-user/shared/safecircle")
from src.messages import score_message
print(score_message("hello"))

{'score': 0.9, 'reasons': ['stub - real model coming']}


In [15]:
import pandas as pd

DATA = "/home/ibm-user/shared/safecircle/data"

In [16]:
sms = pd.read_csv(f"{DATA}/SMSSpamCollection", sep="\t", names=["label", "text"])
extra = pd.read_csv(f"{DATA}/extra_messages.csv")

df = pd.concat([sms, extra], ignore_index=True)
df["y"] = (df["label"].str.strip().str.lower() == "spam").astype(int)

print(df.shape)
print(df["y"].value_counts())
print(f"spam share: {df['y'].mean():.1%}")

(5614, 3)
y
0    4837
1     777
Name: count, dtype: int64
spam share: 13.8%


In [5]:
from sklearn.model_selection import train_test_split

X = df["text"]
y = df["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(len(X_train), "train /", len(X_test), "test")

4491 train / 1123 test


In [6]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000)),
])

pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored if vocabulary is not None.",2
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


In [7]:
from sklearn.metrics import classification_report

pred = pipeline.predict(X_test)
print(classification_report(y_test, pred, target_names=["normal", "spam"]))

              precision    recall  f1-score   support

      normal       0.99      0.99      0.99       968
        spam       0.92      0.94      0.93       155

    accuracy                           0.98      1123
   macro avg       0.96      0.96      0.96      1123
weighted avg       0.98      0.98      0.98      1123



In [8]:
scen = pd.read_csv("/home/ibm-user/shared/safecircle/scenarios/messages.csv")

scen["score"] = pipeline.predict_proba(scen["text"])[:, 1]

print(scen[["user_id", "expected", "score"]])

  user_id expected     score
0    rosa     scam  0.641992
1  walter   normal  0.336300
2   linda     scam  0.696333


In [10]:
import joblib
import os

os.makedirs("/home/ibm-user/shared/safecircle/models", exist_ok=True)
joblib.dump(pipeline, "/home/ibm-user/shared/safecircle/models/message_model.joblib")
print("saved")

saved


In [11]:
import os

p = "/home/ibm-user/shared/safecircle/models/message_model.joblib"
print(os.path.exists(p), os.path.getsize(p), "bytes")

True 370436 bytes


In [13]:
import numpy as np

# 1. Get the vocabulary and the model's weights
feature_names = pipeline.named_steps["tfidf"].get_feature_names_out()
coefs = pipeline.named_steps["clf"].coef_[0]

# 2. Small dictionary to translate model words into plain-English phrases
reason_dict = {
    "gift": "asks for gift cards",
    "cards": "asks for gift cards",
    "locked": "says your account is locked",
    "tell": "asks you to keep it secret",
    # Adding the new words your model actually found:
    "text": "asks you to text a code",
    "it me": "claims to be a family member",
    "account": "mentions your account",
    "your": "uses urgent language"
}

def get_reasons(text, top_n=2):
    # Turn the text into numbers
    tfidf_matrix = pipeline.named_steps["tfidf"].transform([text])
    
    # Multiply the text's word values by the model's weights
    word_scores = tfidf_matrix.toarray()[0] * coefs
    
    # Sort to find the words that pushed the score highest
    top_indices = np.argsort(word_scores)[::-1]
    
    reasons = []
    for i in top_indices:
        # Stop if we hit negative scores (words that make it seem normal) or if we have enough reasons
        if word_scores[i] <= 0 or len(reasons) >= top_n:
            break
        word = feature_names[i]
        
        # Translate to plain English if it's in our dictionary, otherwise use a fallback
        if word in reason_dict:
            if reason_dict[word] not in reasons: # avoid duplicates
                reasons.append(reason_dict[word])
        else:
            reasons.append(f"uses the word '{word}'")
            
    return reasons

# 3. Test it on Rosa, Walter, and Linda
for index, row in scen.iterrows():
    reasons = get_reasons(row["text"])
    print(f"{row['user_id'].capitalize()}: {reasons}")

Rosa: ['asks you to text a code', 'claims to be a family member']
Walter: ['uses urgent language', "uses the word 'for'"]
Linda: ['uses urgent language', 'mentions your account']


In [17]:
import pandas as pd

# 1. Load the extra messages
extra = pd.read_csv("/home/ibm-user/shared/safecircle/data/extra_messages.csv")

# 2. Filter for only the normal texts (label is usually 'ham' in this dataset)
tricky_normals = extra[extra["label"].str.strip().str.lower() == "ham"].copy()

# 3. Score them using your pipeline
tricky_normals["score"] = pipeline.predict_proba(tricky_normals["text"])[:, 1]

# 4. Find the false alarms (score >= 0.5)
false_alarms = tricky_normals[tricky_normals["score"] >= 0.5]

print(f"Total tricky normal texts checked: {len(tricky_normals)}")
print(f"False alarms (accidentally flagged as scam): {len(false_alarms)}\n")

# 5. Show which ones fooled the model
if len(false_alarms) > 0:
    print("Texts that caused a false alarm:")
    for idx, row in false_alarms.iterrows():
        print(f"- Score {row['score']:.2f}: {row['text']}")
else:
    print("Perfect! No false alarms on the tricky texts.")

Total tricky normal texts checked: 12
False alarms (accidentally flagged as scam): 1

Texts that caused a false alarm:
- Score 0.62: Chase: did you make a $45.20 purchase at Target? Reply YES or NO. Chase will never ask for your PIN or password by text.


In [18]:
import time

# Grab 1,000 messages from your main dataframe
sample_texts = df["text"].head(1000).tolist()

start_time = time.time()
# Score all 1,000 messages
_ = pipeline.predict_proba(sample_texts)
end_time = time.time()

speed = end_time - start_time
print(f"Time to score 1,000 messages: {speed:.4f} seconds")

Time to score 1,000 messages: 0.0192 seconds


In [19]:
from sklearn.naive_bayes import MultinomialNB

# 1. Build a new pipeline swapping LogisticRegression for MultinomialNB
nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
    ("clf", MultinomialNB()),
])

# 2. Train it on the exact same data
nb_pipeline.fit(X_train, y_train)

# 3. Print the results to compare with your original model
nb_pred = nb_pipeline.predict(X_test)
print("Multinomial Naive Bayes Results:")
print(classification_report(y_test, nb_pred, target_names=["normal", "spam"]))

Multinomial Naive Bayes Results:
              precision    recall  f1-score   support

      normal       0.95      1.00      0.98       968
        spam       1.00      0.70      0.83       155

    accuracy                           0.96      1123
   macro avg       0.98      0.85      0.90      1123
weighted avg       0.96      0.96      0.96      1123

